In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchvision import transforms
import numpy as np
import os
import json
import random
from PIL import Image
from tqdm import tqdm
import pandas as pd
from dataclasses import dataclass
from collections import Counter
import copy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [3]:
DATASET_ROOT    = "/kaggle/input/datasets/fauzanrafingudin/pexels-dataset/pexels_dataset/"
GLOVE_DIM       = 100
LATENT_DIM      = 256
BATCH_SIZE      = 128 * 10
EPOCHS          = 30
LR              = 1e-3
MAX_SEQ_LEN     = 32
SEED            = 42
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR        = "outputs_multimodal"
LATENT_DIMS     = [4, 8, 16, 32, 64, 128, 256]
IMG_SIZE        = 64
EMBED_DIM       = 128
HIDDEN_DIM      = 256
MIN_FREQ        = 3

torch.backends.cudnn.benchmark = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
@dataclass
class VocabConfig:
    vocab:    list
    word2idx: dict
    idx2word: dict
    size:     int
    pad_idx:  int
    sos_idx:  int
    eos_idx:  int
    unk_idx:  int

def build_vocab(captions, min_freq=3):
    counter = Counter()
    for cap in captions:
        counter.update(cap.lower().split())
    special = ['<pad>', '<sos>', '<eos>', '<unk>']
    words   = [w for w, c in counter.most_common() if c >= min_freq]
    vocab   = special + words
    w2i     = {w: i for i, w in enumerate(vocab)}
    i2w     = {i: w for w, i in w2i.items()}
    print(f"vocab size: {len(vocab)}")
    return VocabConfig(
        vocab    = vocab,
        word2idx = w2i,
        idx2word = i2w,
        size     = len(vocab),
        pad_idx  = w2i['<pad>'],
        sos_idx  = w2i['<sos>'],
        eos_idx  = w2i['<eos>'],
        unk_idx  = w2i['<unk>'],
    )

def encode_caption(caption, vc, max_len=MAX_SEQ_LEN):
    tokens = ['<sos>'] + caption.lower().split()[:max_len-2] + ['<eos>']
    ids    = [vc.word2idx.get(t, vc.unk_idx) for t in tokens]
    ids   += [vc.pad_idx] * (max_len - len(ids))
    return ids[:max_len]

In [5]:
def encode_caption_input(caption, vc, max_len):
    tokens = ['<sos>'] + caption.lower().split()
    tokens = tokens[:max_len]
    ids = [vc.word2idx.get(t, vc.unk_idx) for t in tokens]
    ids += [vc.pad_idx] * (max_len - len(ids))
    return ids[:max_len]

def encode_caption_target(caption, vc, max_len):
    tokens = caption.lower().split() + ['<eos>']
    tokens = tokens[:max_len]
    ids = [vc.word2idx.get(t, vc.unk_idx) for t in tokens]
    ids += [vc.pad_idx] * (max_len - len(ids))
    return ids[:max_len]

In [6]:
class PexelsDataset(Dataset):
    def __init__(self, df, vc, transform=None):
        self.df        = df.reset_index(drop=True)
        self.vc        = vc
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(DATASET_ROOT, row['file'].replace('\\', '/'))
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        caption_ids = encode_caption(row['caption'], self.vc)
        return image, torch.tensor(caption_ids, dtype=torch.long)

def get_dataloaders(vc):
    train_df = pd.read_csv(f"{DATASET_ROOT}/train.csv")
    val_df   = pd.read_csv(f"{DATASET_ROOT}/val.csv")
    test_df  = pd.read_csv(f"{DATASET_ROOT}/test.csv")

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3),
    ])

    dl_kwargs = dict(num_workers=2, pin_memory=True, persistent_workers=True)
    train_dl  = DataLoader(PexelsDataset(train_df, vc, transform), batch_size=BATCH_SIZE, shuffle=True,  **dl_kwargs)
    val_dl    = DataLoader(PexelsDataset(val_df,   vc, transform), batch_size=BATCH_SIZE, shuffle=False, **dl_kwargs)
    test_dl   = DataLoader(PexelsDataset(test_df,  vc, transform), batch_size=BATCH_SIZE, shuffle=False, **dl_kwargs)

    print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")
    return train_dl, val_dl, test_dl

In [7]:
class ImageEncoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),   nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),  # 32x32
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),  # 16x16
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(4),  # 4x4
        )
        self.fc = nn.Linear(128 * 4 * 4, latent_dim)

    def forward(self, x):
        return self.fc(self.conv(x).view(x.size(0), -1))


class ImageDecoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),  # 8x8
            nn.ConvTranspose2d(64, 32,  4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),  # 16x16
            nn.ConvTranspose2d(32, 16,  4, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(),  # 32x32
            nn.ConvTranspose2d(16, 3,   4, stride=2, padding=1), nn.Tanh(),                      # 64x64
        )

    def forward(self, z):
        return self.deconv(self.fc(z).view(-1, 128, 4, 4))

In [8]:
class RNNCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.W_x  = nn.Linear(input_dim, hidden_dim, bias=False)
        self.W_h  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, x, h):
        # h_t = tanh(W_x * x_t + W_h * h_{t-1} + b)
        return torch.tanh(self.W_x(x) + self.W_h(h) + self.bias)


class TextEncoder(nn.Module):
    def __init__(self, vc, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embed      = nn.Embedding(vc.size, embed_dim, padding_idx=vc.pad_idx)
        self.cell       = RNNCell(embed_dim, hidden_dim)
        self.fc         = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embed(x)
        h   = torch.zeros(x.size(0), self.hidden_dim, device=x.device)
        for t in range(emb.size(1)):
            h = self.cell(emb[:, t, :], h)
        return self.fc(h)


class TextDecoder(nn.Module):
    def __init__(self, vc, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.vc = vc
        self.hidden_dim = hidden_dim
        self.embed = nn.Embedding(vc.size, embed_dim, padding_idx=vc.pad_idx)
        self.cell = RNNCell(embed_dim, hidden_dim)
        self.fc_init = nn.Linear(latent_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vc.size)

    def forward(self, z, target_seq):
        h = torch.tanh(self.fc_init(z))
        emb = self.embed(target_seq)
        logits = []
        for t in range(emb.size(1)):
            h = self.cell(emb[:, t, :], h)
            logits.append(self.fc_out(h))
        return torch.stack(logits, dim=1)

    def generate(self, z, max_len):
        self.eval()
        with torch.no_grad():
            h = torch.tanh(self.fc_init(z))
            token = torch.full((z.size(0),), self.vc.sos_idx, dtype=torch.long, device=z.device)

            result = []
            for _ in range(max_len):
                emb = self.embed(token)
                h = self.cell(emb, h)
                logits = self.fc_out(h)
                token = logits.argmax(dim=-1)
                result.append(token)
                if (token == self.vc.eos_idx).all():
                    break

            result = torch.stack(result, dim=1)

        # Decode ke teks
        sentences = []
        for seq in result.cpu().numpy():
            words = []
            for idx in seq:
                if idx == self.vc.eos_idx:
                    break
                if idx not in (self.vc.pad_idx, self.vc.sos_idx):
                    words.append(self.vc.idx2word.get(idx, '<unk>'))
            sentences.append(' '.join(words))

        return sentences

In [9]:
class LatentCaptionDataset(Dataset):
    def __init__(self, dataloader, img_enc, vc, device, max_len):
        self.samples = []
        img_enc.eval()
        with torch.no_grad():
            for imgs, captions in tqdm(dataloader, desc="Building latent dataset"):
                imgs = imgs.to(device)
                latents = img_enc(imgs).cpu()
                for i in range(len(imgs)):
                    cap_ids = captions[i].cpu()
                    self.samples.append((cap_ids, latents[i]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cap_ids, latent = self.samples[idx]
        return cap_ids, latent

In [10]:
def precompute_image_tensors(dataloader):
    print("Precomputing image tensors into RAM")
    all_imgs, all_caps = [], []
    for imgs, caps in tqdm(dataloader):
        all_imgs.append(imgs)
        all_caps.extend(caps)
    all_imgs = torch.cat(all_imgs, dim=0)  # (N, 3, 32, 32)
    print(f"  Done. Tensor shape: {all_imgs.shape} | RAM ~{all_imgs.nbytes/1e6:.1f} MB")
    return all_imgs, all_caps


def make_fast_dataloaders(train_dl, val_dl, test_dl):
    train_imgs, train_caps = precompute_image_tensors(train_dl)
    val_imgs,   val_caps   = precompute_image_tensors(val_dl)
    test_imgs,  test_caps  = precompute_image_tensors(test_dl)

    # Wrapper supaya caption tetap bisa diakses
    class CachedDataset(Dataset):
        def __init__(self, imgs, caps):
            self.imgs = imgs
            self.caps = caps
        def __len__(self):
            return len(self.imgs)
        def __getitem__(self, idx):
            return self.imgs[idx], self.caps[idx]

    fast_train_dl = DataLoader(
        CachedDataset(train_imgs, train_caps),
        batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=0
    )
    fast_val_dl = DataLoader(
        CachedDataset(val_imgs, val_caps),
        batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0
    )
    fast_test_dl = DataLoader(
        CachedDataset(test_imgs, test_caps),
        batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0
    )
    return fast_train_dl, fast_val_dl, fast_test_dl


In [13]:
def decode_ids_batch(ids_batch, vc):
    sentences = []
    for seq in ids_batch:
        words = []
        for idx in seq:
            idx = idx.item() if isinstance(idx, torch.Tensor) else idx
            if idx == vc.eos_idx:
                break
            if idx not in (vc.pad_idx, vc.sos_idx):
                words.append(vc.idx2word.get(idx, '<unk>'))
        sentences.append(' '.join(words))
    return sentences

In [11]:
def train_first(latent_dim, train_dl, val_dl, test_dl, save_dir):
    ssim_metric = StructuralSimilarityIndexMeasure(
        kernel_size=11,sigma=1.5, data_range=1.0
    ).to(DEVICE)

    enc = ImageEncoder(latent_dim).to(DEVICE)
    dec = ImageDecoder(latent_dim).to(DEVICE)

    if torch.cuda.device_count() > 1:
        print(f"  Using {torch.cuda.device_count()} GPUs")
        enc = nn.DataParallel(enc)
        dec = nn.DataParallel(dec)
    
    opt = optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=LR)
    scheduler = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

    train_losses, val_losses_log = [], []

    all_kernels = {
        "enc_conv1": [], "enc_conv2": [], "enc_conv3": [],
        "dec_deconv1": [], "dec_deconv2": [], "dec_deconv3": [],
        "dec_fc": [],
    }
    all_latents = {"z": [], "labels": []}
    all_recons = {"orig": [], "recon": [], "epoch": []}

    for epoch in range(EPOCHS):
        # Train
        enc.train(); dec.train()
        epoch_loss = 0.0
        for imgs, _ in train_dl:
            imgs = imgs.to(DEVICE)
            loss = F.mse_loss(dec(enc(imgs)), imgs)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_dl))
        scheduler.step()

        # Val
        enc.eval(); dec.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, _ in val_dl:
                imgs = imgs.to(DEVICE)
                val_loss += F.mse_loss(dec(enc(imgs)), imgs).item()
        val_loss /= len(val_dl)
        val_losses_log.append(val_loss)

        # Collect Everything
        _enc = enc.module if isinstance(enc, nn.DataParallel) else enc
        _dec = dec.module if isinstance(dec, nn.DataParallel) else dec

        # 1. Kernel Encoder
        all_kernels["enc_conv1"].append(_enc.conv[0].weight.detach().cpu().numpy())
        all_kernels["enc_conv2"].append(_enc.conv[4].weight.detach().cpu().numpy())
        all_kernels["enc_conv3"].append(_enc.conv[8].weight.detach().cpu().numpy())

        # 2. Kernel Decoder
        all_kernels["dec_deconv1"].append(_dec.deconv[0].weight.detach().cpu().numpy())
        all_kernels["dec_deconv2"].append(_dec.deconv[3].weight.detach().cpu().numpy())
        all_kernels["dec_deconv3"].append(_dec.deconv[6].weight.detach().cpu().numpy())
        all_kernels["dec_fc"].append(_dec.fc.weight.detach().cpu().numpy())

        # 3. Latent vectors + labels
        zs, ys = [], []
        with torch.no_grad():
            for imgs, labels in test_dl:
                imgs = imgs.to(DEVICE)
                zs.append(_enc(imgs).cpu().numpy())
                ys.append(np.array(labels))
        all_latents["z"].append(np.concatenate(zs, axis=0))
        all_latents["labels"].append(np.concatenate(ys, axis=0))

        # 4. Sample rekonstruksi
        with torch.no_grad():
            sample_imgs, _ = next(iter(test_dl))
            sample_imgs = sample_imgs[:8].to(DEVICE)
            sample_recon = dec(enc(sample_imgs)).cpu()
        all_recons["orig"].append(sample_imgs.cpu().numpy())
        all_recons["recon"].append(sample_recon.numpy())
        all_recons["epoch"].append(epoch)

        print(f"  [{latent_dim:>4d}] epoch {epoch+1:>3d}/{EPOCHS}  "
              f"train_loss={train_losses[-1]:.5f}  val_loss={val_loss:.5f}", end="\r")

    print()

    # Save ALL
    np.savez_compressed(
        f"{save_dir}/all_data_dim{latent_dim}.npz",
        # Losses
        train_losses=np.array(train_losses),
        val_losses=np.array(val_losses_log),
        # Kernels
        enc_conv1=np.array(all_kernels["enc_conv1"]),
        enc_conv2=np.array(all_kernels["enc_conv2"]),
        enc_conv3=np.array(all_kernels["enc_conv3"]),
        dec_deconv1=np.array(all_kernels["dec_deconv1"]),
        dec_deconv2=np.array(all_kernels["dec_deconv2"]),
        dec_deconv3=np.array(all_kernels["dec_deconv3"]),
        dec_fc=np.array(all_kernels["dec_fc"]),
        # Latents
        latent_z=np.array(all_latents["z"]),
        latent_labels=np.array(all_latents["labels"]),
        # Reconstructions
        recon_orig=np.array(all_recons["orig"]),
        recon_recon=np.array(all_recons["recon"]),
        recon_epochs=np.array(all_recons["epoch"]),
    )

    # Save Final Model
    torch.save({
        "encoder":    _enc.state_dict(),
        "decoder":    _dec.state_dict(),
        "latent_dim": latent_dim,
    }, f"{save_dir}/model_dim{latent_dim}.pt")

    # Evaluate Final
    enc.eval(); dec.eval()
    mse_total, ssim_total, n = 0.0, 0.0, 0
    sample_orig = sample_recon = sample_labels = None


    with torch.no_grad():
        for imgs, _ in test_dl:
            imgs   = imgs.to(DEVICE)
            recons = dec(enc(imgs))
            mse_total += F.mse_loss(recons, imgs, reduction='sum').item()
            ssim_total += ssim_metric(recons, imgs).item() * len(imgs)
            n += len(imgs)

            if sample_orig is None:
                sample_orig   = imgs[:8].cpu()
                sample_recon  = recons[:8].cpu()
                sample_labels = list(labels[:8])

    mse  = mse_total / (n * 3 * IMG_SIZE * IMG_SIZE)
    ssim = ssim_total / n
    print(f"  [{latent_dim:>4d}] MSE={mse:.5f}  SSIM={ssim:.4f}  saved.")

    return {
        "latent_dim": latent_dim,
        "mse":        round(mse, 6),
        "ssim":       round(ssim, 4),
        "params":     sum(p.numel() for p in enc.parameters()) +
                      sum(p.numel() for p in dec.parameters()),
        "train_losses": train_losses,
        "val_losses":   val_losses_log,
    }, sample_orig, sample_recon, sample_labels

In [12]:
def train_second(latent_dim, train_dl, val_dl, test_dl, vc, save_dir_root):
    save_dir = f"{save_dir_root}/train_second"
    os.makedirs(save_dir, exist_ok=True)

    # Load & freeze ImageEncoder
    checkpoint = torch.load(
        f"{save_dir_root}/train_first/model_dim{latent_dim}.pt",
        map_location=DEVICE
    )
    img_enc = ImageEncoder(latent_dim).to(DEVICE)
    img_enc.load_state_dict(checkpoint["encoder"])
    img_enc.eval()
    for p in img_enc.parameters():
        p.requires_grad = False

    # Build latent datasets (token IDs + image latent)
    print(f"  [{latent_dim:>4d}] Generating latent targets")
    train_latent_ds = LatentCaptionDataset(train_dl, img_enc, vc, DEVICE, MAX_SEQ_LEN)
    val_latent_ds   = LatentCaptionDataset(val_dl,   img_enc, vc, DEVICE, MAX_SEQ_LEN)
    test_latent_ds  = LatentCaptionDataset(test_dl,  img_enc, vc, DEVICE, MAX_SEQ_LEN)

    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=2, persistent_workers=True)
    train_latent_dl = DataLoader(train_latent_ds, shuffle=True,  **dl_kw)
    val_latent_dl   = DataLoader(val_latent_ds,   shuffle=False, **dl_kw)
    test_latent_dl  = DataLoader(test_latent_ds,  shuffle=False, **dl_kw)

    # Init TextEncoder RNN
    txt_enc = TextEncoder(
        vc=vc,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        latent_dim=latent_dim
    ).to(DEVICE)

    if torch.cuda.device_count() > 1:
        txt_enc = nn.DataParallel(txt_enc)

    opt = optim.Adam(txt_enc.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

    best_val_loss = float("inf")
    best_state = None
    train_losses, val_losses_log = [], []

    for epoch in range(EPOCHS):
        # Train
        txt_enc.train()
        epoch_loss = 0.0
        for cap_ids, latent_target in train_latent_dl:
            cap_ids = cap_ids.to(DEVICE)
            latent_target = latent_target.to(DEVICE)

            pred = txt_enc(cap_ids)
            loss = F.mse_loss(pred, latent_target)

            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_latent_dl))
        scheduler.step()

        # Val
        txt_enc.eval()
        val_loss = 0.0
        with torch.no_grad():
            for cap_ids, latent_target in val_latent_dl:
                cap_ids = cap_ids.to(DEVICE)
                latent_target = latent_target.to(DEVICE)
                val_loss += F.mse_loss(txt_enc(cap_ids), latent_target).item()
        val_loss /= len(val_latent_dl)
        val_losses_log.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            _txt = txt_enc.module if isinstance(txt_enc, nn.DataParallel) else txt_enc
            best_state = copy.deepcopy(_txt.state_dict())

        print(f"  [{latent_dim:>4d}] epoch {epoch+1:>3d}/{EPOCHS}  "
              f"train={train_losses[-1]:.5f}  val={val_loss:.5f}", end="\r")

    print()

    # Save Loss Log
    np.savez_compressed(
        f"{save_dir}/loss_log_dim{latent_dim}.npz",
        train_losses=np.array(train_losses),
        val_losses=np.array(val_losses_log),
    )

    # Load best state & evaluate on test
    _txt = txt_enc.module if isinstance(txt_enc, nn.DataParallel) else txt_enc
    if best_state:
        _txt.load_state_dict(best_state)
    _txt.eval()

    # Save best model
    torch.save({
        "text_encoder": _txt.state_dict(),
        "latent_dim":    latent_dim,
        "vc":            vc,
    }, f"{save_dir}/model_dim{latent_dim}.pt")

    test_loss = 0.0
    with torch.no_grad():
        for cap_ids, latent_target in test_latent_dl:
            cap_ids = cap_ids.to(DEVICE)
            latent_target = latent_target.to(DEVICE)
            test_loss += F.mse_loss(_txt(cap_ids), latent_target).item()
    test_loss /= len(test_latent_dl)

    print(f"  [{latent_dim:>4d}] best_val={best_val_loss:.5f}  test={test_loss:.5f}  saved.")

    return {
        "latent_dim":    latent_dim,
        "best_val_loss": round(best_val_loss, 6),
        "test_loss":     round(test_loss, 6),
        "params":        sum(p.numel() for p in _txt.parameters()),
        "train_losses":  train_losses,
        "val_losses":    val_losses_log,
    }

In [28]:
def train_third(latent_dim, train_dl, val_dl, test_dl, vc, save_dir_root):
    save_dir = f"{save_dir_root}/train_third"
    os.makedirs(save_dir, exist_ok=True)

    # Load frozen ImageEncoder
    checkpoint_img = torch.load(
        f"{save_dir_root}/train_first/model_dim{latent_dim}.pt",
        map_location=DEVICE
    )
    img_enc = ImageEncoder(latent_dim).to(DEVICE)
    img_enc.load_state_dict(checkpoint_img["encoder"])
    img_enc.eval()
    for p in img_enc.parameters():
        p.requires_grad = False

    # Build latent dataset (image latent + caption token IDs)
    print(f"  [{latent_dim:>4d}] Generating latent targets")
    train_latent_ds = LatentCaptionDataset(train_dl, img_enc, vc, DEVICE, MAX_SEQ_LEN)
    val_latent_ds   = LatentCaptionDataset(val_dl,   img_enc, vc, DEVICE, MAX_SEQ_LEN)
    test_latent_ds  = LatentCaptionDataset(test_dl,  img_enc, vc, DEVICE, MAX_SEQ_LEN)

    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=2, persistent_workers=True)
    train_latent_dl = DataLoader(train_latent_ds, shuffle=True,  **dl_kw)
    val_latent_dl   = DataLoader(val_latent_ds,   shuffle=False, **dl_kw)
    test_latent_dl  = DataLoader(test_latent_ds,  shuffle=False, **dl_kw)

    # Init TextDecoder
    txt_dec = TextDecoder(
        vc=vc,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        latent_dim=latent_dim
    ).to(DEVICE)
    
    if torch.cuda.device_count() > 1:
        txt_dec = nn.DataParallel(txt_dec)
    
    opt = optim.Adam(txt_dec.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
    criterion = nn.CrossEntropyLoss(ignore_index=vc.pad_idx)

    best_val_loss = float("inf")
    best_state = None
    train_losses, val_losses_log = [], []

    for epoch in range(EPOCHS):
        # Train
        txt_dec.train()
        epoch_loss = 0.0
        for cap_ids, img_latent in train_latent_dl:
            cap_ids = cap_ids.to(DEVICE)
            img_latent = img_latent.to(DEVICE)

            # Pisah input & target
            cap_input  = cap_ids[:, :-1]   # <sos>
            cap_target = cap_ids[:, 1:]     # kata

            # Forward
            logits = txt_dec(img_latent, cap_input)  # (B, seq_len-1, vocab_size)

            # Loss
            loss = criterion(
                logits.reshape(-1, vc.size),
                cap_target.reshape(-1)
            )

            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_latent_dl))
        scheduler.step()

        # Val
        txt_dec.eval()
        val_loss = 0.0
        with torch.no_grad():
            for cap_ids, img_latent in val_latent_dl:
                cap_ids = cap_ids.to(DEVICE)
                img_latent = img_latent.to(DEVICE)

                cap_input  = cap_ids[:, :-1]
                cap_target = cap_ids[:, 1:]

                logits = txt_dec(img_latent, cap_input)
                loss = criterion(
                    logits.reshape(-1, vc.size),
                    cap_target.reshape(-1)
                )
                val_loss += loss.item()
        val_loss /= len(val_latent_dl)
        val_losses_log.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            _txt = txt_dec.module if isinstance(txt_dec, nn.DataParallel) else txt_dec
            best_state = copy.deepcopy(_txt.state_dict())

        print(f"  [{latent_dim:>4d}] epoch {epoch+1:>3d}/{EPOCHS}  "
              f"train={train_losses[-1]:.5f}  val={val_loss:.5f}", end="\r")

    print()

    # Save Loss Log
    np.savez_compressed(
        f"{save_dir}/loss_log_dim{latent_dim}.npz",
        train_losses=np.array(train_losses),
        val_losses=np.array(val_losses_log),
    )

    # Load best state
    _txt = txt_dec.module if isinstance(txt_dec, nn.DataParallel) else txt_dec
    if best_state:
        _txt.load_state_dict(best_state)
    _txt.eval()

    # Save best model
    torch.save({
        "text_decoder": _txt.state_dict(),
        "latent_dim":   latent_dim,
        "vc":           vc,
    }, f"{save_dir}/model_dim{latent_dim}.pt")

    # Evaluate on test set
    test_loss = 0.0
    with torch.no_grad():
        for cap_ids, img_latent in test_latent_dl:
            cap_ids = cap_ids.to(DEVICE)
            img_latent = img_latent.to(DEVICE)

            cap_input  = cap_ids[:, :-1]
            cap_target = cap_ids[:, 1:]

            logits = _txt(img_latent, cap_input)
            loss = criterion(
                logits.reshape(-1, vc.size),
                cap_target.reshape(-1)
            )
            test_loss += loss.item()
    test_loss /= len(test_latent_dl)

    # Generate sample captions
    print(f"\n  [{latent_dim:>4d}] Sample generations:")
    with torch.no_grad():
        for cap_ids, img_latent in test_latent_dl:
            img_latent = img_latent[:5].to(DEVICE)

            # Ground truth
            gt_cap_ids = cap_ids[:5]
            gt_texts = decode_ids_batch(gt_cap_ids, vc)

            # Generated
            gen_texts = _txt.generate(img_latent, max_len=MAX_SEQ_LEN)

            for gt, gen in zip(gt_texts, gen_texts):
                print(f"    GT:  {gt}")
                print(f"    GEN: {gen}")
                print()
            break

    print(f"  [{latent_dim:>4d}] best_val={best_val_loss:.5f}  test={test_loss:.5f}  saved.")

    return {
        "latent_dim":    latent_dim,
        "best_val_loss": round(best_val_loss, 6),
        "test_loss":     round(test_loss, 6),
        "params":        sum(p.numel() for p in _txt.parameters()),
        "train_losses":  train_losses,
        "val_losses":    val_losses_log,
    }

In [15]:
def run_experiment(save_dir):
    train_df_raw = pd.read_csv(f"{DATASET_ROOT}/train.csv")
    vc           = build_vocab(train_df_raw['caption'].tolist(), min_freq=MIN_FREQ)
    train_dl, val_dl, test_dl = get_dataloaders(vc)
    
    # Precompute images ke RAM untuk eliminasi CPU bottleneck
    fast_train_dl, fast_val_dl, fast_test_dl = make_fast_dataloaders(
        train_dl, val_dl, test_dl
    )

    # Train First: CNN Autoencoder
    save_dir_first = f"{save_dir}/train_first"
    os.makedirs(save_dir_first, exist_ok=True)
    
    results_first, all_samples = [], {}
    for dim in LATENT_DIMS:
        print(f"\nTraining CNN autoencoder latent_dim={dim}")
        res, s_orig, s_recon, s_labels = train_first(
            dim, fast_train_dl, fast_val_dl, fast_test_dl, save_dir_first
        )
        results_first.append(res)
        all_samples[dim] = (s_orig, s_recon, s_labels)
    
    with open(f"{save_dir_first}/results.json", "w") as f:
        json.dump([{k: v for k, v in r.items() if k not in ('train_losses', 'val_losses')}
                    for r in results_first], f, indent=2)

    # Train Second: FNN TextEncoder
    save_dir_second = f"{save_dir}/train_second"
    os.makedirs(save_dir_second, exist_ok=True)
    
    results_second = []
    for dim in LATENT_DIMS:
        print(f"\nTraining FNN text encoder latent_dim={dim}")
        res = train_second(dim, fast_train_dl, fast_val_dl, fast_test_dl, vc, save_dir)
        results_second.append(res)
    
    with open(f"{save_dir_second}/results.json", "w") as f:
        json.dump([{k: v for k, v in r.items() if k not in ('train_losses', 'val_losses')}
                    for r in results_second], f, indent=2)

    save_dir_third = f"{save_dir}/train_third"
    os.makedirs(save_dir_third, exist_ok=True)
    
    results_third = []
    for dim in LATENT_DIMS:
        print(f"\nTraining FNN text encoder latent_dim={dim}")
        res = train_third(dim, fast_train_dl, fast_val_dl, fast_test_dl, vc, save_dir)
        results_third.append(res)
    
    with open(f"{save_dir_third}/results.json", "w") as f:
        json.dump([{k: v for k, v in r.items() if k not in ('train_losses', 'val_losses')}
                    for r in results_third], f, indent=2)
    
    return results_first, all_samples, results_second, results_third


In [16]:
def plot_results(results_first, all_samples, results_second=None, results_third=None, save_dir=SAVE_DIR):

    dims = [r["latent_dim"] for r in results_first]

    # Loss Curve per Epoch - CNN Autoencoder
    if results_first and "train_losses" in results_first[0]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Loss Curve per Epoch - CNN Autoencoder", fontsize=13, fontweight='bold')
        colors = plt.cm.tab10(np.linspace(0, 1, len(LATENT_DIMS)))

        for r, c in zip(results_first, colors):
            dim = r["latent_dim"]
            axes[0].plot(r["train_losses"], label=f"z={dim}", color=c, linewidth=1.5)
            axes[1].plot(r["val_losses"],   label=f"z={dim}", color=c, linewidth=1.5)

        for ax, title in zip(axes, ["Train Loss", "Val Loss"]):
            ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
            ax.set_title(title); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{save_dir}/loss_curve_first.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: loss_curve_first.png")

    # Loss Curve per Epoch - RNN TextEncoder (train_second)
    if results_second and "train_losses" in results_second[0]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Loss Curve per Epoch - RNN TextEncoder (Text → Latent)", fontsize=13, fontweight='bold')
        colors = plt.cm.tab10(np.linspace(0, 1, len(LATENT_DIMS)))

        for r, c in zip(results_second, colors):
            dim = r["latent_dim"]
            axes[0].plot(r["train_losses"], label=f"z={dim}", color=c, linewidth=1.5)
            axes[1].plot(r["val_losses"],   label=f"z={dim}", color=c, linewidth=1.5)

        for ax, title in zip(axes, ["Train Loss", "Val Loss"]):
            ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
            ax.set_title(title); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{save_dir}/loss_curve_second.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: loss_curve_second.png")

    # Loss Curve per Epoch - RNN TextDecoder (train_third)
    if results_third and "train_losses" in results_third[0]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Loss Curve per Epoch - RNN TextDecoder (Latent → Text)", fontsize=13, fontweight='bold')
        colors = plt.cm.tab10(np.linspace(0, 1, len(LATENT_DIMS)))

        for r, c in zip(results_third, colors):
            dim = r["latent_dim"]
            axes[0].plot(r["train_losses"], label=f"z={dim}", color=c, linewidth=1.5)
            axes[1].plot(r["val_losses"],   label=f"z={dim}", color=c, linewidth=1.5)

        for ax, title in zip(axes, ["Train Loss", "Val Loss"]):
            ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-Entropy Loss")
            ax.set_title(title); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{save_dir}/loss_curve_third.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: loss_curve_third.png")

    # MSE + SSIM vs Latent Dim (train_first)
    mses  = [r["mse"]  for r in results_first]
    ssims = [r["ssim"] for r in results_first]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Analisis Kapasitas Ruang Laten - CNN Autoencoder", fontsize=13, fontweight='bold')

    axes[0].plot(dims, mses, 'o-', color='steelblue', linewidth=2, markersize=8)
    axes[0].set_xlabel("Dimensi Ruang Laten"); axes[0].set_ylabel("MSE")
    axes[0].set_title("Reconstruction Error (↓ lebih baik)")
    axes[0].set_xscale('log', base=2); axes[0].grid(True, alpha=0.3)
    for d, m in zip(dims, mses):
        axes[0].annotate(f'{m:.4f}', (d, m), textcoords="offset points",
                         xytext=(0, 9), fontsize=8, ha='center')

    axes[1].plot(dims, ssims, 's-', color='coral', linewidth=2, markersize=8)
    axes[1].set_xlabel("Dimensi Ruang Laten"); axes[1].set_ylabel("SSIM")
    axes[1].set_title("Structural Similarity (↑ lebih baik)")
    axes[1].set_xscale('log', base=2); axes[1].grid(True, alpha=0.3)
    for d, s in zip(dims, ssims):
        axes[1].annotate(f'{s:.3f}', (d, s), textcoords="offset points",
                         xytext=(0, 9), fontsize=8, ha='center')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/kapasitas_ruang_laten.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: kapasitas_ruang_laten.png")

    # Val/Test Loss vs Latent Dim - TextEncoder (train_second)
    if results_second:
        val_losses_s  = [r["best_val_loss"] for r in results_second]
        test_losses_s = [r["test_loss"]     for r in results_second]

        fig, ax = plt.subplots(figsize=(8, 4))
        fig.suptitle("Train Second - TextEncoder → Latent (Text → z)",
                     fontsize=13, fontweight='bold')
        ax.plot(dims, val_losses_s,  'o-',  color='mediumpurple', lw=2, ms=8, label='Val Loss (best)')
        ax.plot(dims, test_losses_s, 's--', color='darkorange',   lw=2, ms=8, label='Test Loss')
        ax.set_xlabel("Dimensi Ruang Laten"); ax.set_ylabel("MSE Loss")
        ax.set_title("Text → Image Latent (↓ lebih baik)")
        ax.set_xscale('log', base=2); ax.grid(True, alpha=0.3); ax.legend()

        for d, v, t in zip(dims, val_losses_s, test_losses_s):
            ax.annotate(f'{v:.4f}', (d, v), textcoords="offset points",
                        xytext=(0,  9), fontsize=7, ha='center', color='mediumpurple')
            ax.annotate(f'{t:.4f}', (d, t), textcoords="offset points",
                        xytext=(0,-13), fontsize=7, ha='center', color='darkorange')

        plt.tight_layout()
        plt.savefig(f"{save_dir}/train_second_loss.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: train_second_loss.png")

    # Val/Test Loss vs Latent Dim - TextDecoder (train_third)
    if results_third:
        val_losses_t  = [r["best_val_loss"] for r in results_third]
        test_losses_t = [r["test_loss"]     for r in results_third]

        fig, ax = plt.subplots(figsize=(8, 4))
        fig.suptitle("Train Third - TextDecoder (Latent → Text)",
                     fontsize=13, fontweight='bold')
        ax.plot(dims, val_losses_t,  'o-',  color='teal',        lw=2, ms=8, label='Val Loss (best)')
        ax.plot(dims, test_losses_t, 's--', color='tomato',      lw=2, ms=8, label='Test Loss')
        ax.set_xlabel("Dimensi Ruang Laten"); ax.set_ylabel("Cross-Entropy Loss")
        ax.set_title("Image Latent → Caption (↓ lebih baik)")
        ax.set_xscale('log', base=2); ax.grid(True, alpha=0.3); ax.legend()

        for d, v, t in zip(dims, val_losses_t, test_losses_t):
            ax.annotate(f'{v:.4f}', (d, v), textcoords="offset points",
                        xytext=(0,  9), fontsize=7, ha='center', color='teal')
            ax.annotate(f'{t:.4f}', (d, t), textcoords="offset points",
                        xytext=(0,-13), fontsize=7, ha='center', color='tomato')

        plt.tight_layout()
        plt.savefig(f"{save_dir}/train_third_loss.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("Saved: train_third_loss.png")

    # Rekonstruksi per Dim
    if all_samples:
        available_dims = [d for d in LATENT_DIMS if d in all_samples]
        n_dims = len(available_dims)

        if n_dims > 0:
            fig = plt.figure(figsize=(20, n_dims * 2.8 + 1))
            fig.suptitle("Rekonstruksi Citra per Dimensi Ruang Laten (Pexels)",
                         fontsize=13, fontweight='bold')
            gs = gridspec.GridSpec(n_dims * 2, 8, hspace=0.05, wspace=0.05)

            for row_idx, dim in enumerate(available_dims):
                s_orig, s_recon, s_labels = all_samples[dim]
                for col in range(min(8, len(s_orig))):
                    ax_o = fig.add_subplot(gs[row_idx*2, col])
                    img_o = (s_orig[col] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
                    ax_o.imshow(img_o); ax_o.axis('off')
                    if col == 0:
                        ax_o.set_ylabel(f"z={dim}\nOri", fontsize=8,
                                        rotation=0, labelpad=42, va='center')

                    ax_r = fig.add_subplot(gs[row_idx*2+1, col])
                    img_r = (s_recon[col] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
                    ax_r.imshow(img_r); ax_r.axis('off')
                    if col == 0:
                        ax_r.set_ylabel("Rekon", fontsize=8,
                                        rotation=0, labelpad=42, va='center')

            plt.savefig(f"{save_dir}/rekonstruksi_per_dim.png", dpi=150, bbox_inches='tight')
            plt.close()
            print("Saved: rekonstruksi_per_dim.png")
    else:
        print("Skipping rekonstruksi plot (all_samples kosong)")

    # Tabel Perbandingan (1-3 tabel)
    n_tables = 1 + (1 if results_second else 0) + (1 if results_third else 0)
    fig, axes = plt.subplots(1, n_tables, figsize=(10 * n_tables, 3.5))
    if n_tables == 1:
        axes = [axes]

    # Tabel 1: CNN Autoencoder
    ax1 = axes[0]
    ax1.axis('off')
    rows1 = [[str(r["latent_dim"]), f"{r['mse']:.5f}",
              f"{r['ssim']:.4f}", f"{r['params']:,}"] for r in results_first]
    t1 = ax1.table(cellText=rows1,
                   colLabels=["Latent Dim", "MSE ↓", "SSIM ↑", "Params"],
                   cellLoc='center', loc='center')
    t1.auto_set_font_size(False); t1.set_fontsize(10); t1.scale(1.2, 1.9)

    best_mse_idx  = mses.index(min(mses))
    best_ssim_idx = ssims.index(max(ssims))
    for col in range(4):
        t1[(best_mse_idx + 1, col)].set_facecolor('#d4edda')
    if best_ssim_idx != best_mse_idx:
        for col in range(4):
            t1[(best_ssim_idx + 1, col)].set_facecolor('#cce5ff')
    ax1.set_title("Train First - CNN Autoencoder", fontweight='bold', pad=10)

    # Tabel 2: TextEncoder
    if results_second:
        ax2 = axes[1]
        ax2.axis('off')
        rows2 = [[str(r["latent_dim"]), f"{r['best_val_loss']:.5f}",
                  f"{r['test_loss']:.5f}", f"{r['params']:,}"] for r in results_second]
        t2 = ax2.table(cellText=rows2,
                       colLabels=["Latent Dim", "Val Loss ↓", "Test Loss ↓", "Params"],
                       cellLoc='center', loc='center')
        t2.auto_set_font_size(False); t2.set_fontsize(10); t2.scale(1.2, 1.9)

        best_val_idx = [r["best_val_loss"] for r in results_second].index(
                        min(r["best_val_loss"] for r in results_second))
        for col in range(4):
            t2[(best_val_idx + 1, col)].set_facecolor('#d4edda')
        ax2.set_title("Train Second - TextEncoder (Text→Latent)", fontweight='bold', pad=10)

    # Tabel 3: TextDecoder
    if results_third:
        ax3 = axes[-1]  # index terakhir
        ax3.axis('off')
        rows3 = [[str(r["latent_dim"]), f"{r['best_val_loss']:.5f}",
                  f"{r['test_loss']:.5f}", f"{r['params']:,}"] for r in results_third]
        t3 = ax3.table(cellText=rows3,
                       colLabels=["Latent Dim", "Val Loss ↓", "Test Loss ↓", "Params"],
                       cellLoc='center', loc='center')
        t3.auto_set_font_size(False); t3.set_fontsize(10); t3.scale(1.2, 1.9)

        best_val_idx = [r["best_val_loss"] for r in results_third].index(
                        min(r["best_val_loss"] for r in results_third))
        for col in range(4):
            t3[(best_val_idx + 1, col)].set_facecolor('#d4edda')
        ax3.set_title("Train Third - TextDecoder (Latent→Text)", fontweight='bold', pad=10)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/tabel_perbandingan.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: tabel_perbandingan.png")

In [17]:
if __name__ == "__main__":
    import shutil

    results_first, all_samples, results_second, results_third = run_experiment(SAVE_DIR)
    plot_results(results_first, all_samples, results_second, results_third, save_dir=SAVE_DIR)

    print("\n=== HASIL EKSPERIMEN - TRAIN FIRST ===")
    print(f"{'Dim':>6} {'MSE':>10} {'SSIM':>8} {'Params':>12}")
    print("-" * 42)
    
    for r in results_first:
        print(f"{r['latent_dim']:>6} {r['mse']:>10.5f} {r['ssim']:>8.4f} {r['params']:>12,}")

    print("\n=== HASIL EKSPERIMEN - TRAIN SECOND ===")
    print(f"{'Dim':>6} {'Val Loss':>12} {'Test Loss':>12} {'Params':>12}")
    print("-" * 48)
    for r in results_second:
        print(f"{r['latent_dim']:>6} {r['best_val_loss']:>12.5f} {r['test_loss']:>12.5f} {r['params']:>12,}")

    print("\n=== HASIL EKSPERIMEN - TRAIN THIRD ===")
    print(f"{'Dim':>6} {'Val Loss':>12} {'Test Loss':>12} {'Params':>12}")
    print("-" * 48)
    for r in results_third:
        print(f"{r['latent_dim']:>6} {r['best_val_loss']:>12.5f} {r['test_loss']:>12.5f} {r['params']:>12,}")
        
    shutil.make_archive(SAVE_DIR, 'zip', SAVE_DIR)
    print(f"\nOutput: {SAVE_DIR}.zip")

vocab size: 5488
train=39743 val=4972 test=5007
Precomputing image tensors into RAM


 38%|███▊      | 12/32 [01:55<02:36,  7.82s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 75%|███████▌  | 24/32 [03:40<00:58,  7.33s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 32/32 [04:50<00:00,  9.07s/it]


  Done. Tensor shape: torch.Size([39743, 3, 64, 64]) | RAM ~1953.4 MB
Precomputing image tensors into RAM


100%|██████████| 4/4 [00:35<00:00,  8.88s/it]


  Done. Tensor shape: torch.Size([4972, 3, 64, 64]) | RAM ~244.4 MB
Precomputing image tensors into RAM


...

  [  64] epoch  30/30  train=2.57383  val=2.72285

  [  64] Sample generations:
    GT:  vibrant rooster stands proudly on a wooden table amidst lush green garden foliage.
    GEN: a detailed close-up of a green iguana basking in the sun on a sunny day.

    GT:  a joyful young woman holding a white rabbit outdoors in the evening sun.
    GEN: a serene scene of a herd of african elephants grazing in a lush green pasture under a clear blue sky.

    GT:  a serene tabby cat with striking eyes lounging outside. captures feline grace.
    GEN: a detailed close-up of a green iguana resting on a branch in a natural setting.

    GT:  cute pigs exploring a rustic farmyard with natural sunlight.
    GEN: a serene scene of a herd of african elephants grazing in a lush green pasture under a clear sky.

    GT:  a detailed close-up of a green praying mantis poised on a leaf with a blurred natural background.
    GEN: a serene scene of a herd of african elephants grazing in a lush green pasture un

Building latent dataset: 100%|██████████| 4/4 [00:00<00:00,  7.41it/s]


  [ 128] epoch  30/30  train=2.55117  val=2.71005

  [ 128] Sample generations:
    GT:  vibrant rooster stands proudly on a wooden table amidst lush green garden foliage.
    GEN: a detailed close-up of a green iguana basking in the sun on a sunny day.

    GT:  a joyful young woman holding a white rabbit outdoors in the evening sun.
    GEN: a herd of african elephants walking through a lush green pasture under a clear blue sky.

    GT:  a serene tabby cat with striking eyes lounging outside. captures feline grace.
    GEN: a detailed close-up of a green iguana basking on a tree branch in a natural setting.

    GT:  cute pigs exploring a rustic farmyard with natural sunlight.
    GEN: a detailed close-up of a brown bear in its natural habitat, showcasing its natural habitat.

    GT:  a detailed close-up of a green praying mantis poised on a leaf with a blurred natural background.
    GEN: a serene scene of a sheep grazing in a lush green pasture under a clear blue sky.

  [ 128] b

Building latent dataset: 100%|██████████| 4/4 [00:00<00:00,  7.46it/s]


  [ 256] epoch  30/30  train=2.55498  val=2.71588

  [ 256] Sample generations:
    GT:  vibrant rooster stands proudly on a wooden table amidst lush green garden foliage.
    GEN: a detailed close-up of a green iguana basking in the sun on a sunny day.

    GT:  a joyful young woman holding a white rabbit outdoors in the evening sun.
    GEN: a herd of african elephants walking through a dusty landscape during the day.

    GT:  a serene tabby cat with striking eyes lounging outside. captures feline grace.
    GEN: a detailed close-up of a green iguana resting on a tree branch in a natural setting.

    GT:  cute pigs exploring a rustic farmyard with natural sunlight.
    GEN: a detailed close-up of a black cat with striking blue eyes and a red collar, looking directly at the camera.

    GT:  a detailed close-up of a green praying mantis poised on a leaf with a blurred natural background.
    GEN: a serene scene of a white horse grazing in a lush green pasture under a clear sky.

  [